In [ ]:
# 变量表调试 Notebook

逐步调试 `create_tag_table_with_tags` 接口，排查"变量地址正常但名称/表名为空"的问题。

**调试流程：**
1. 加载 TIA API
2. 启动 TIA Portal 并创建项目
3. 添加 PLC 设备
4. 构造测试变量并调用 `create_tag_table_with_tags`
5. 回读变量表内容，验证写入结果
6. 清理会话

## 第 1 步：环境准备 —— 添加工程路径并加载 TIA API

In [1]:
# 若你刚修改过 openness/tia_core.py，这一格可强制重载模块
import importlib
import tia_core as tia_core

# 重置 API 已加载标记，并重载模块
try:
    tia_core._api_loaded = False
except Exception:
    pass
importlib.reload(tia_core)

print("✅ tia_core 已重载，后续将使用最新的 DLL 加载逻辑")

import sys
import os

# 将项目根目录加入 Python 路径，使 openness 包可导入
PROJECT_ROOT = r"E:\PlcProject\Code\PLC\SCDW"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

API_DIR = r"E:\PlcProject\SoftWares\Siemens\Automation\Portal V17\PublicAPI\V17"

from openness.tia_core import load_tia_api, set_default_api_dir
set_default_api_dir(API_DIR)
load_tia_api(API_DIR)

print("✅ TIA API 加载完成")

✅ tia_core 已重载，后续将使用最新的 DLL 加载逻辑
✅ TIA API 加载完成


## 第 2 步：启动 TIA Portal 并创建测试项目

In [2]:
import os
import sys
import traceback
import clr  # type: ignore

from tia_core import load_tia_api, set_default_api_dir
from tia_core import start_tia_portal, create_project

PROJECT_ROOT_DIR = r"E:\PlcProject\Projects"
PROJECT_NAME = "TagDebugTest"

# 参考 init_tia_project：先 set_default_api_dir，再 load_tia_api，再启动/建项目
# 不修改 tia_core 启动代码，只在此单元补齐 Contract 依赖搜索路径
try:
    set_default_api_dir(API_DIR)

    tia_root = r"E:\PlcProject\SoftWares\Siemens\Automation\Portal V17"
    bin_dir = os.path.join(tia_root, "Bin")
    bin_public_api_dir = os.path.join(bin_dir, "PublicAPI")

    probe_dirs = [API_DIR, os.path.dirname(API_DIR), tia_root, bin_dir, bin_public_api_dir]

    for p in probe_dirs:
        if p and os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)

    env_path = os.environ.get("PATH", "")
    for p in probe_dirs:
        if p and os.path.isdir(p) and p not in env_path:
            os.environ["PATH"] = p + os.pathsep + os.environ.get("PATH", "")
            env_path = os.environ["PATH"]

    # 显式预加载 Contract，避免在导入 Siemens.Engineering 类型时才失败
    contract_dll = os.path.join(bin_public_api_dir, "Siemens.Engineering.Contract.dll")
    if os.path.isfile(contract_dll):
        clr.AddReference(contract_dll)

    load_tia_api(API_DIR)

    tia = start_tia_portal(with_ui=True)
    project = create_project(tia, PROJECT_ROOT_DIR, PROJECT_NAME, overwrite=True)

    print("✅ TIA 项目已创建")
    print(f"  项目名称：{PROJECT_NAME}")
    print(f"  项目路径：{os.path.join(PROJECT_ROOT_DIR, PROJECT_NAME)}")
    print("  TIA UI：有界面")
except Exception as exc:
    print(f"❌ 创建项目失败：{exc}")
    print(traceback.format_exc())

✅ TIA 项目已创建
  项目名称：TagDebugTest
  项目路径：E:\PlcProject\Projects\TagDebugTest
  TIA UI：有界面


## 第 3 步：添加 PLC 设备（CPU 1214C DC/DC/DC）

In [3]:
from openness.tia_hardware import add_plc_device

CPU_ORDER_NUMBER = "OrderNumber:6ES7 214-1BG40-0XB0/V4.5"
DEVICE_NAME      = "PLC_1"

device, plc_sw = add_plc_device(project, CPU_ORDER_NUMBER, DEVICE_NAME, DEVICE_NAME)
print(f"✅ PLC 设备已添加：{DEVICE_NAME}  订货号：{CPU_ORDER_NUMBER}")

✅ PLC 设备已添加：PLC_1  订货号：OrderNumber:6ES7 214-1BG40-0XB0/V4.5


## 第 4 步：构造测试变量

准备一批覆盖常见情况的测试变量，包括：
- 普通 Bool 输入/输出
- 整型字地址（IW/QW/MW）
- 带中文注释的变量
- 可能导致问题的边界情况（名称含特殊字符、注释为空等）

In [4]:
from openness.tia_tags import TagSpec

test_tags = [
    # 普通 Bool 输入
    TagSpec(name="启动按钮",      data_type="Bool", logical_address="%I0.0",  comment="启动信号"),
    TagSpec(name="停止按钮",      data_type="Bool", logical_address="%I0.1",  comment="停止信号"),
    TagSpec(name="急停",          data_type="Bool", logical_address="%I0.2",  comment="急停按钮"),
    # Bool 输出
    TagSpec(name="运行指示灯",    data_type="Bool", logical_address="%Q0.0",  comment="运行状态指示"),
    TagSpec(name="故障指示灯",    data_type="Bool", logical_address="%Q0.1",  comment=""),
    # 字型地址
    TagSpec(name="模拟量输入_1",  data_type="Word", logical_address="%IW64",  comment="模拟量通道1"),
    TagSpec(name="模拟量输入_2",  data_type="Word", logical_address="%IW66",  comment="模拟量通道2"),
    TagSpec(name="模拟量输出_1",  data_type="Word", logical_address="%QW80",  comment="AO通道1"),
    # 中间继电器/标志位
    TagSpec(name="运行标志",      data_type="Bool", logical_address="%M0.0",  comment=""),
    TagSpec(name="故障标志",      data_type="Bool", logical_address="%M0.1",  comment="系统故障"),
    TagSpec(name="计数值",        data_type="Int",  logical_address="%MW10",  comment="脉冲计数"),
    # 注释为空的变量（触发过空注释问题的典型场景）
    TagSpec(name="备用_I1_0",     data_type="Bool", logical_address="%I1.0",  comment=""),
    TagSpec(name="备用_I1_1",     data_type="Bool", logical_address="%I1.1",  comment=""),
]

print(f"准备写入 {len(test_tags)} 个变量：")
for t in test_tags:
    print(f"  {t.name:<16}  {t.data_type:<6}  {t.logical_address:<8}  '{t.comment}'")

准备写入 13 个变量：
  启动按钮              Bool    %I0.0     '启动信号'
  停止按钮              Bool    %I0.1     '停止信号'
  急停                Bool    %I0.2     '急停按钮'
  运行指示灯             Bool    %Q0.0     '运行状态指示'
  故障指示灯             Bool    %Q0.1     ''
  模拟量输入_1           Word    %IW64     '模拟量通道1'
  模拟量输入_2           Word    %IW66     '模拟量通道2'
  模拟量输出_1           Word    %QW80     'AO通道1'
  运行标志              Bool    %M0.0     ''
  故障标志              Bool    %M0.1     '系统故障'
  计数值               Int     %MW10     '脉冲计数'
  备用_I1_0           Bool    %I1.0     ''
  备用_I1_1           Bool    %I1.1     ''


## 第 5 步：调用 `create_tag_table_with_tags` 写入变量表

逐条写入并捕获每个变量的异常，定位到底哪条失败。

In [5]:
from openness.tia_tags import create_tag_table, add_tag
from openness.tia_hardware import find_device, get_plc_software

TABLE_NAME = "调试变量表"

# 先刷新 plc_sw 句柄，避免使用已释放对象（EngineeringObjectDisposedException）
try:
    device = find_device(project, DEVICE_NAME)
    plc_sw = get_plc_software(device)
    _ = plc_sw.TagTableGroup  # 触发一次访问，提前验证句柄是否有效
except Exception as e:
    raise RuntimeError(
        "plc_sw 句柄已失效，请先重新运行第 2 步和第 3 步，再执行本单元。"
    ) from e

# 创建变量表
tag_table = create_tag_table(plc_sw, TABLE_NAME)
print(f"✅ 变量表 '{TABLE_NAME}' 已创建\n")

# 逐条写入，独立捕获每条异常，便于精确定位问题
success_list = []
fail_list = []

for spec in test_tags:
    try:
        add_tag(tag_table, spec.name, spec.data_type, spec.logical_address, spec.comment)
        success_list.append(spec)
        print(f"  ✅ 写入成功：{spec.name:<16}  {spec.data_type:<6}  {spec.logical_address}")
    except Exception as e:
        fail_list.append((spec, str(e)))
        print(f"  ❌ 写入失败：{spec.name:<16}  {spec.data_type:<6}  {spec.logical_address}  →  {e}")

print(f"\n总计：成功 {len(success_list)} / 失败 {len(fail_list)}")

✅ 变量表 '调试变量表' 已创建

  ✅ 写入成功：启动按钮              Bool    %I0.0
  ✅ 写入成功：停止按钮              Bool    %I0.1
  ✅ 写入成功：急停                Bool    %I0.2
  ✅ 写入成功：运行指示灯             Bool    %Q0.0
  ✅ 写入成功：故障指示灯             Bool    %Q0.1
  ✅ 写入成功：模拟量输入_1           Word    %IW64
  ✅ 写入成功：模拟量输入_2           Word    %IW66
  ✅ 写入成功：模拟量输出_1           Word    %QW80
  ✅ 写入成功：运行标志              Bool    %M0.0
  ✅ 写入成功：故障标志              Bool    %M0.1
  ✅ 写入成功：计数值               Int     %MW10
  ✅ 写入成功：备用_I1_0           Bool    %I1.0
  ✅ 写入成功：备用_I1_1           Bool    %I1.1

总计：成功 13 / 失败 0


## 第 6 步：回读变量表内容，验证实际写入结果

通过 TIA Openness API 直接遍历变量表中的每个 Tag，打印其名称、类型、地址，与预期对比，找出缺失或信息为空的条目。

In [6]:
print(f"{'序号':<4} {'名称':<18} {'类型':<8} {'地址':<10} {'注释'}")
print("-" * 65)

anomalies = []  # 记录异常条目

for i, tag in enumerate(tag_table.Tags):
    try:
        name    = str(tag.Name)
        dtype   = str(tag.DataTypeName)
        address = str(tag.LogicalAddress)
        try:
            comment = str(tag.Comment.Items.Item(0).Text)
        except Exception:
            comment = "(读取注释失败)"

        # 检测异常：名称或地址为空
        is_anomaly = (name.strip() == "" or address.strip() == "")
        marker = " ⚠️" if is_anomaly else ""
        print(f"{i:<4} {name:<18} {dtype:<8} {address:<10} {comment}{marker}")

        if is_anomaly:
            anomalies.append({"index": i, "name": name, "dtype": dtype, "address": address})
    except Exception as e:
        print(f"{i:<4} (读取失败: {e})")

print(f"\n变量表共 {i + 1} 条，其中异常条目 {len(anomalies)} 条")
if anomalies:
    print("\n❌ 异常条目详情：")
    for a in anomalies:
        print(f"  index={a['index']}  name='{a['name']}'  type={a['dtype']}  address='{a['address']}'")

序号   名称                 类型       地址         注释
-----------------------------------------------------------------
0    启动按钮               Bool     %I0.0      (读取注释失败)
1    停止按钮               Bool     %I0.1      (读取注释失败)
2    急停                 Bool     %I0.2      (读取注释失败)
3    运行指示灯              Bool     %Q0.0      (读取注释失败)
4    故障指示灯              Bool     %Q0.1      (读取注释失败)
5    模拟量输入_1            Word     %IW64      (读取注释失败)
6    模拟量输入_2            Word     %IW66      (读取注释失败)
7    模拟量输出_1            Word     %QW80      (读取注释失败)
8    运行标志               Bool     %M0.0      (读取注释失败)
9    故障标志               Bool     %M0.1      (读取注释失败)
10   计数值                Int      %MW10      (读取注释失败)
11   备用_I1_0            Bool     %I1.0      (读取注释失败)
12   备用_I1_1            Bool     %I1.1      (读取注释失败)

变量表共 13 条，其中异常条目 0 条


## 第 7 步：对比预期与实际结果

检查哪些预期变量在实际表中缺失，哪些多余。

In [ ]:
# 读取实际变量表中所有名称
actual_names   = {str(tag.Name) for tag in tag_table.Tags}
expected_names = {spec.name for spec in test_tags}

missing = expected_names - actual_names
extra   = actual_names - expected_names

print(f"预期变量数：{len(expected_names)}，实际变量数：{len(actual_names)}")

if missing:
    print(f"\n⚠️  缺失的变量（{len(missing)} 个）：")
    for n in sorted(missing):
        spec = next(s for s in test_tags if s.name == n)
        print(f"  {n:<18}  {spec.data_type:<6}  {spec.logical_address}")
else:
    print("\n✅ 所有预期变量均已写入")

if extra:
    print(f"\nℹ️  多余条目（{len(extra)} 个，可能是空名称占位）：")
    for n in sorted(extra):
        print(f"  '{n}'")

## 第 8 步：单步调试 `add_tag` 内部行为

如果上面发现了异常，在这里单独重现问题变量的写入过程，并检查 `Tags.Create` 返回对象的所有属性，确认是写入时就缺失还是回读时丢失。

In [ ]:
# 用第 6 步中发现的某个缺失/异常变量来重现问题
# 如未发现问题，可把下面的值替换成 missing 集合中的任意一个
PROBE_NAME    = "模拟量输入_1"
PROBE_DTYPE   = "Word"
PROBE_ADDRESS = "%IW64"
PROBE_COMMENT = "模拟量通道1"

TABLE_PROBE = "调试变量表_单步"
probe_table = create_tag_table(plc_sw, TABLE_PROBE)

print(f"正在创建变量：name={PROBE_NAME!r}  type={PROBE_DTYPE}  addr={PROBE_ADDRESS}")
raw_tag = probe_table.Tags.Create(PROBE_NAME, PROBE_DTYPE, PROBE_ADDRESS)
print(f"\nTags.Create 返回对象类型：{type(raw_tag)}")

# 立即回读创建后的属性
print(f"  .Name           = {raw_tag.Name!r}")
print(f"  .DataTypeName   = {raw_tag.DataTypeName!r}")
print(f"  .LogicalAddress = {raw_tag.LogicalAddress!r}")

# 设置注释
if PROBE_COMMENT:
    try:
        raw_tag.Comment.Items.Item(0).Text = PROBE_COMMENT
        print(f"  .Comment 写入    = {PROBE_COMMENT!r}")
    except Exception as ce:
        print(f"  .Comment 写入失败：{ce}")

# 再次回读（确认注释写入后名称/地址是否变化）
print(f"\n写入注释后再次回读：")
print(f"  .Name           = {raw_tag.Name!r}")
print(f"  .DataTypeName   = {raw_tag.DataTypeName!r}")
print(f"  .LogicalAddress = {raw_tag.LogicalAddress!r}")

## 第 9 步：清理会话，关闭 TIA Portal

In [ ]:
from openness.tia_core import save_project
from openness.tia_core import stop_tia_portal

# 保存项目（方便在 TIA 界面中手动确认结果）
save_project(project)
print("✅ 项目已保存")

# 若不需要继续在 TIA 界面观察，取消注释以下两行关闭 Portal
# tia.Dispose()
# print("✅ TIA Portal 已关闭")